In [2]:
import pandas as pd
import numpy as np

# Load one file first to see all columns
df_test = pd.read_parquet('fhvhv_tripdata_2024-10.parquet')
print(df_test.shape)
print(df_test.columns.tolist())

(20028282, 24)
['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']


In [3]:
df_test_2025 = pd.read_parquet('fhvhv_tripdata_2025-01.parquet')
print(df_test_2025.columns.tolist())
print(df_test_2025.shape)

['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag', 'cbd_congestion_fee']
(20405666, 25)


In [5]:
#columns to keep for 2024 files (no cbd_congestion_fee)
cols_2024 = [
    'hvfhs_license_num', 'pickup_datetime', 'dropoff_datetime',
    'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time',
    'base_passenger_fare', 'tolls', 'congestion_surcharge',
    'airport_fee', 'tips', 'driver_pay'
]

#loading the data for Q4 2024
october = pd.read_parquet('fhvhv_tripdata_2024-10.parquet', columns=cols_2024).sample(n=500000, random_state=42)
november = pd.read_parquet('fhvhv_tripdata_2024-11.parquet', columns=cols_2024).sample(n=500000, random_state=42)
december = pd.read_parquet('fhvhv_tripdata_2024-12.parquet', columns=cols_2024).sample(n=500000, random_state=42)

#adding period label and cbd_congestion_fee as 0 for 2024
for df in [october, november, december]:
    df['cbd_congestion_fee'] = 0.0
    df['period'] = 'before'

print("Q4 2024 loaded")
print(october.shape, november.shape, december.shape)

Q4 2024 loaded
(500000, 15) (500000, 15) (500000, 15)


In [8]:
#columns to keep for 2025 files (including cbd_congestion_fee)
cols_2025 = [
    'hvfhs_license_num', 'pickup_datetime', 'dropoff_datetime',
    'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time',
    'base_passenger_fare', 'tolls', 'congestion_surcharge',
    'airport_fee', 'tips', 'driver_pay', 'cbd_congestion_fee'
]

#loading the sample Q1 2025 data
january = pd.read_parquet('fhvhv_tripdata_2025-01.parquet', columns=cols_2025).sample(n=500000, random_state=42)
february = pd.read_parquet('fhvhv_tripdata_2025-02.parquet', columns=cols_2025).sample(n=500000, random_state=42)
march = pd.read_parquet('fhvhv_tripdata_2025-03.parquet', columns=cols_2025).sample(n=500000, random_state=42)

#adding period label 
for df in [january, february, march]:
    df['period'] = 'after'

print("Q1 2025 loaded")
print(january.shape, february.shape, march.shape)

Q1 2025 loaded
(500000, 15) (500000, 15) (500000, 15)


In [9]:
#combining all 6 months of data:
df = pd.concat([october, november, december, january, february, march], ignore_index=True)

print("Combined shape:", df.shape)
print("\nPeriod Distribution:")
print(df['period'].value_counts())
print("\nUber vs Lyft:")
print(df['hvfhs_license_num'].value_counts())
print("\nNull values:")
print(df.isnull().sum())


Combined shape: (3000000, 15)

Period Distribution:
period
before    1500000
after     1500000
Name: count, dtype: int64

Uber vs Lyft:
hvfhs_license_num
HV0003    2237054
HV0005     762946
Name: count, dtype: int64

Null values:
hvfhs_license_num       0
pickup_datetime         0
dropoff_datetime        0
PULocationID            0
DOLocationID            0
trip_miles              0
trip_time               0
base_passenger_fare     0
tolls                   0
congestion_surcharge    0
airport_fee             0
tips                    0
driver_pay              0
cbd_congestion_fee      0
period                  0
dtype: int64


In [10]:
print("Trip miles stats:")
print(df['trip_miles'].describe())
print("\nDriver pay stats:")
print(df['driver_pay'].describe())
print("\nBase passenger fare stats:")
print(df['base_passenger_fare'].describe())
print("\nNegative values check:")
print("Negative driver pay:", (df['driver_pay'] < 0).sum())
print("Negative fares:", (df['base_passenger_fare'] < 0).sum())
print("Zero trip miles:", (df['trip_miles'] == 0).sum())

Trip miles stats:
count    3.000000e+06
mean     5.027782e+00
std      5.869710e+00
min      0.000000e+00
25%      1.530000e+00
50%      2.950000e+00
75%      6.300000e+00
max      3.322100e+02
Name: trip_miles, dtype: float64

Driver pay stats:
count    3.000000e+06
mean     2.020754e+01
std      1.727404e+01
min     -3.756000e+01
25%      8.980000e+00
50%      1.512000e+01
75%      2.550000e+01
max      8.659500e+02
Name: driver_pay, dtype: float64

Base passenger fare stats:
count    3.000000e+06
mean     2.637762e+01
std      2.314474e+01
min     -1.873000e+01
25%      1.251000e+01
50%      1.960000e+01
75%      3.157000e+01
max      1.121590e+03
Name: base_passenger_fare, dtype: float64

Negative values check:
Negative driver pay: 44
Negative fares: 221
Zero trip miles: 412


In [12]:
#removing bad rows
df_clean = df[
    (df['driver_pay'] > 0) & 
    (df['base_passenger_fare'] > 0) &
    (df['trip_miles'] > 0) &
    (df['trip_time'] > 0)
]

#remove extreme outliers (99.9th percentile)
fare_cap = df_clean['base_passenger_fare'].quantile(0.999)
pay_cap = df_clean['driver_pay'].quantile(0.999)
miles_cap = df_clean['trip_miles'].quantile(0.999)

df_clean = df_clean[
    (df_clean['base_passenger_fare'] <= fare_cap) & 
    (df_clean['driver_pay'] <= pay_cap) &
    (df_clean['trip_miles'] <= miles_cap)
]

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original rows: 3000000
Clean rows: 2990468
Rows removed: 9532


In [13]:
#key calculated columns
#Driver take rate - what % of passenger fare goes to driver 
df_clean['driver_take_rate'] = df_clean['driver_pay'] / df_clean['base_passenger_fare']

#Total fees paid by passenger (congestion + surcharge + airport)
df_clean['total_fees'] = (df_clean['congestion_surcharge'] +
                          df_clean['cbd_congestion_fee'] +
                          df_clean['airport_fee'] +
                          df_clean['tolls'])

#trip duration in minutes (easier to interpret than seconds)
df_clean['trip_time_mins'] = df_clean['trip_time'] / 60

#Earnings per mile
df_clean['earnings_per_mile'] = df_clean['driver_pay'] / df_clean['trip_miles']

#label Uber vs Lyft
df_clean['platform'] = df_clean['hvfhs_license_num'].map(
    {
        'HV0003': 'Uber',
        'HV0005': 'Lyft'
    }
)

print(df_clean[['driver_take_rate', 'total_fees', 'trip_time_mins', 
                'earnings_per_mile', 'platform']].describe())

       driver_take_rate    total_fees  trip_time_mins  earnings_per_mile
count      2.990468e+06  2.990468e+06    2.990468e+06       2.990468e+06
mean       7.767523e-01  2.599414e+00    1.955715e+01       5.525714e+00
std        9.833657e-01  4.443813e+00    1.368614e+01       7.859737e+00
min        2.671756e-03  0.000000e+00    1.666667e-02       1.808786e-02
25%        6.214953e-01  0.000000e+00    9.850000e+00       3.658281e+00
50%        7.416434e-01  0.000000e+00    1.598333e+01       4.753555e+00
75%        8.859294e-01  3.470000e+00    2.533333e+01       6.171462e+00
max        1.588000e+03  7.317000e+01    2.556000e+02       3.653000e+03


In [14]:
# Take rate should be between 0 and 2 (allowing for tips pushing it above 1)
df_clean = df_clean[
    (df_clean['driver_take_rate'] > 0) &
    (df_clean['driver_take_rate'] <= 2) &
    (df_clean['earnings_per_mile'] <= 50)  # cap at $50/mile
]

print("Rows after take rate filter:", len(df_clean))
print("\nDriver take rate stats:")
print(df_clean['driver_take_rate'].describe())
print("\nEarnings per mile stats:")
print(df_clean['earnings_per_mile'].describe())

Rows after take rate filter: 2984690

Driver take rate stats:
count    2.984690e+06
mean     7.724325e-01
std      2.089847e-01
min      2.671756e-03
25%      6.213135e-01
50%      7.413276e-01
75%      8.851002e-01
max      2.000000e+00
Name: driver_take_rate, dtype: float64

Earnings per mile stats:
count    2.984690e+06
mean     5.437205e+00
std      3.069964e+00
min      1.808786e-02
25%      3.657230e+00
50%      4.751244e+00
75%      6.163931e+00
max      5.000000e+01
Name: earnings_per_mile, dtype: float64


In [15]:
# Save to CSV
df_clean.to_csv('tlc_trips_clean.csv', index=False)
print("Saved! Rows:", len(df_clean))
print("Columns:", df_clean.columns.tolist())

Saved! Rows: 2984690
Columns: ['hvfhs_license_num', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'cbd_congestion_fee', 'period', 'driver_take_rate', 'total_fees', 'trip_time_mins', 'earnings_per_mile', 'platform']


In [17]:
import os
size = os.path.getsize('tlc_trips_clean.csv') / (1024**3)
print(f"CSV size: {size:.2f} GB")

CSV size: 0.45 GB


In [19]:
#Splitting csv into 5 chunks 
chunk_size = 500000
chunks = [df_clean[i:i+chunk_size] for i in range(0, len(df_clean), chunk_size)]

for i, chunk in enumerate(chunks):
    filename = f'tlc_trips_chunk_{i+1}.csv'
    chunk.to_csv(filename, index=False)
    print(f"Saved {filename} - {len(chunk)} rows - {os.path.getsize(filename)/(1024**2):.1f} MB")

Saved tlc_trips_chunk_1.csv - 500000 rows - 78.3 MB
Saved tlc_trips_chunk_2.csv - 500000 rows - 78.2 MB
Saved tlc_trips_chunk_3.csv - 500000 rows - 78.2 MB
Saved tlc_trips_chunk_4.csv - 500000 rows - 77.7 MB
Saved tlc_trips_chunk_5.csv - 500000 rows - 77.7 MB
Saved tlc_trips_chunk_6.csv - 484690 rows - 75.4 MB
